In [1]:
# Import libraries and configure Ollama

import autogen
from autogen import AssistantAgent, LLMConfig

# Select Ollama model
MODEL = "qwen3:1.7b"

# Ollama configuration
llm_config = LLMConfig({"model": MODEL, "api_type": "ollama", "client_host": "http://localhost:11434",
                        "stream": False}, temperature = 0.03)


In [2]:
# Define the task
task = """Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting."""


# Create Writer Agent

writer = AssistantAgent(name = "Writer", 
                        system_message = """You are a professional content writer.
                        Your responsibilities:
                        1. Write engaging and concise blog posts.
                        2. Always include an attractive title.
                        3. Follow the given word limit.
                        4. Improve your writing based on feedback.
                        5. Provide only the final polished blog post.""",
                        llm_config = llm_config)
    

In [3]:
# Generate the initial blog post
reply = writer.generate_reply(messages = [{"content": task, 
                                            "role": "user"}])

print(reply)

{'content': '**Empowering Tomorrow: How Hope AI is Shaping the Future**  \n\nHope AI is revolutionizing industries with cutting-edge AI solutions. From healthcare to finance, they drive innovation and efficiency. Their tools empower businesses to tackle complex challenges, boost productivity, and grow sustainably. Ethical AI and customer success stories highlight their commitment to progress. Join the future—transform your operations today! 🌟  \n\n(98 words)', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


In [4]:
# Create Critic Agent

critic = AssistantAgent(name = "Critic", is_termination_msg = lambda x: (x.get("content", "").find("TERMINATE") >= 0),
                        llm_config = llm_config, 
                        system_message = """You are a professional content critic.
                        Your responsibilities:
                        1. Review the writer's blog post.
                        2. Identify grammatical mistakes.
                        3. Check clarity and readability.
                        4. Identify areas for improvement.
                        5. Provide constructive feedback.""")
                        

In [5]:
# Start Writer-Critic conversation

res = critic.initiate_chat(recipient = writer, message = task, max_turns = 2, summary_method = "last_msg")

Critic (to Writer):

Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting.

--------------------------------------------------------------------------------
Writer (to Critic):

**Empowering Tomorrow: How Hope AI is Shaping the Future**  
Hope AI is revolutionizing industries with cutting-edge AI solutions. From healthcare to finance, they blend innovation with practicality. Our natural language processing and machine learning tools drive efficiency and accuracy. Whether boosting productivity or predicting trends, Hope AI turns challenges into opportunities. Join us in building a smarter future—discover how we’re redefining possibilities. 🌟  

(98 words)

--------------------------------------------------------------------------------
Critic (to Writer):

**Title:** *Empowering Tomorrow: How Hope AI is Shaping the Fu

In [6]:
# Create SEO Reviewer
SEO_reviewer = AssistantAgent(name = "SEO_Reviewer", llm_config = llm_config, 
                              system_message = """You are an SEO reviewer.
                              Your responsibilities:
                              1. Check SEO optimization.
                              2. Suggest relevant keywords.
                              3. Improve search engine visibility.
                              Provide feedback within 3 bullet points.
                              Begin your review by stating your role.""")

In [7]:
# Create Legal Reviewer
legal_reviewer = AssistantAgent(name = "Legal_Reviewer", llm_config = llm_config,
                                system_message = """You are a legal content reviewer.
                                Your responsibilities:
                                1. Identify potentially misleading claims.
                                2. Check for potential legal concerns.
                                3. Suggest improvements where necessary.
                            
                                Provide feedback within 3 bullet points.
                                Begin your review by stating your role.""")

In [8]:
# Create Ethics Reviewer
ethic_reviewer = AssistantAgent(name = "Ethic_Reviewer", llm_config = llm_config,
                                system_message = """You are an ethics reviewer.
                                Your responsibilities:
                                1. Check whether the content is ethically appropriate.
                                2. Identify biased or misleading statements.
                                3. Ensure responsible communication.
                            
                                Provide feedback within 3 bullet points.
                                Begin your review by stating your role.""")

In [9]:
# Create Meta Reviewer
meta_reviewer = AssistantAgent(name = "Meta_Reviewer", llm_config = llm_config,
                               system_message = """You are a meta reviewer.
                               Your responsibilities:
                               1. Review feedback from all reviewers.
                               2. Combine their suggestions.
                               3. Identify the most important improvements.
                               4. Provide clear final recommendations.""")

In [10]:
# Define reflection message
def reflection_message(recipient, messages, sender, config):
    return f"""
    Review the following blog post.

    Blog post:
    {recipient.chat_messages_for_summary(sender)[-1]['content']}

    Provide your feedback based on your assigned role."""

In [11]:
# Define nested review chats
review_chats = [{"recipient": SEO_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "SEO_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},
                
                {"recipient": legal_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "legal_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},

                {"recipient": ethic_reviewer, "messages": reflection_message,
                 "summary_method": "reflection_with_llm", "summary_args": {
                     "summary_prompt": """Return the review as a JSON object only: {
                     "Reviewer": "ethics_reviewer", "Review": "Your feedback"}"""}, "max_turns": 1},

                {"recipient": meta_reviewer, "messages": """Aggregate feedback from all reviewers
                and provide final suggestions for improving the blog post.""", "max_turns": 1}]
                 

In [12]:
# Register nested chats
critic.register_nested_chats(review_chats, trigger=writer)

In [13]:
# Execute the complete multi-agent workflow
res = critic.initiate_chat(recipient=writer, message=task, max_turns = 2, summary_method = "last_msg")

Critic (to Writer):

Write a concise but engaging blog post about Hope AI.
Requirements:
1. Keep the blog post within 100 words.
2. Include an attractive title.
3. Use simple and engaging language.
4. Make the content informative and interesting.

--------------------------------------------------------------------------------
Writer (to Critic):

**Empowering Tomorrow: How Hope AI is Shaping the Future**  
Hope AI is revolutionizing industries with cutting-edge AI solutions. From healthcare to finance, they blend innovation with practicality. Our natural language processing and machine learning tools drive efficiency and accuracy. Whether boosting productivity or predicting trends, Hope AI turns challenges into opportunities. Join us in building a smarter future—discover how we’re redefining possibilities. 🌟  

(98 words)

--------------------------------------------------------------------------------

********************************************************************************
S

In [14]:
# Display the final output
print("Final Blog Post:")
print(res.summary)

Final Blog Post:
**Revolutionizing Industries: How Hope AI is Shaping the Future**  
Hope AI leads the charge in AI innovation, powering healthcare, finance, and beyond. Our tools boost efficiency, predict trends, and drive accuracy. From smart analytics to personalized solutions, we turn challenges into opportunities. Ready to transform your business? Discover how Hope AI redefines possibilities—*today*. 🚀  

(98 words)
